# Lab 8: Cleaning and Preparing a Login Table
Course 314341 AI in Cybersecurity and Cloud Computing, Ajloun National University. Dr. Mohammed Tawfik.

Run the cells from top to bottom in Google Colab (Runtime → Run all, or Shift+Enter cell by cell). Read the comments: lines that start with # explain the code.

**Data:** `dirty_logins.csv`, a small login table made for this course with mistakes on purpose (values invented). It is loaded from the course GitHub: nothing to upload.

## 1. Load and look for problems

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/kmkholm/ai-cybersecurity-cloud-course/main/labs/data/dirty_logins.csv"
df = pd.read_csv(url)
print(df.shape)
df.head(10)

In [ ]:
print(df.isna().sum())                                   # missing values per column
print("exact duplicates:", df.duplicated().sum())
print(df["country"].value_counts(dropna=False))          # how many spellings?
print(df["failed_attempts"].describe())                  # any impossible values?

## 2. Clean: duplicates, impossible values, formats, missing values

In [ ]:
df = df.drop_duplicates()
df = df[df["failed_attempts"] >= 0]                      # a count cannot be negative
df["country"] = (df["country"].str.strip().str.upper()
                 .replace({"JORDAN": "JO", "NETHERLANDS": "NL"}))
df["country"] = df["country"].fillna("UNKNOWN")         # in security, "unknown" beats guessing
print(df.shape)
print(df["country"].value_counts())

## 3. Time: convert Amman local time to UTC and build time features

In [ ]:
local = pd.to_datetime(df["time_local"])
df["time_utc"] = local.dt.tz_localize("Asia/Amman").dt.tz_convert("UTC")
df["hour_local"] = local.dt.hour
df["night"] = df["hour_local"].between(0, 5).astype(int)
df[["time_local", "time_utc", "hour_local", "night"]].head()

## 4. Outliers: error or attack? Look before you delete!

In [ ]:
print(df.sort_values("failed_attempts", ascending=False)[["time_local", "user", "country", "failed_attempts", "label"]].head(8))

## 5. Turn everything into numbers (one-hot) and split by time

In [ ]:
df = df.sort_values("time_utc")
X = pd.get_dummies(df[["failed_attempts", "hour_local", "night", "country"]], columns=["country"], dtype=int)
y = df["label"]
cut = int(len(df) * 0.7)                                 # oldest 70% = training, newest 30% = test
X_train, X_test, y_train, y_test = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]
print("train until", df["time_utc"].iloc[cut - 1], "| test from", df["time_utc"].iloc[cut])
print(y_train.value_counts(), y_test.value_counts(), sep="\n")
X.head()

## 6. A first detector on the clean data

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import confusion_matrix
tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
print(export_text(tree, feature_names=list(X.columns)))
print(confusion_matrix(y_test, tree.predict(X_test), labels=["attack", "normal"]))